# Question Answering

Потребуются следующие данные:

- [MIRAGE](https://github.com/nlpai-lab/MIRAGE) (`../data/mirage/`)
- [SQuAD](https://huggingface.co/datasets/rajpurkar/squad)

Все данные предполагаются в директории `../data/`.

Для работы потребуется установить:

```bash
pip install bert-score datasets rank-bm25 sentence-transformers accelerate
```

In [1]:
import os

os.environ['HF_HUB_DISABLE_XET'] = '1'

import json
import pickle
import re
import string
from collections import Counter, defaultdict
from typing import Callable

import bert_score.utils as bs_utils
import numpy as np
import pandas as pd
import torch
import transformers
from bert_score import score as bert_score_fn
from packaging import version
from rank_bm25 import BM25Okapi
from sentence_transformers import CrossEncoder
from tqdm import tqdm
from transformers import (
    AutoModelForCausalLM,
    AutoModelForQuestionAnswering,
    AutoTokenizer,
    GPT2Tokenizer,
    RobertaTokenizer,
    pipeline,
)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
CACHE_DIR = '../data/cache/hw5'
os.makedirs(CACHE_DIR, exist_ok=True)


def cache_load(name: str):
    path = os.path.join(CACHE_DIR, f'{name}.pkl')
    if os.path.exists(path):
        with open(path, 'rb') as f:
            return pickle.load(f)
    return None


def cache_save(name: str, obj) -> None:
    path = os.path.join(CACHE_DIR, f'{name}.pkl')
    with open(path, 'wb') as f:
        pickle.dump(obj, f)


print(f'Устройство: {DEVICE}')
print(f'Путь до кэша: {CACHE_DIR}')

d:\GitHub\itmo-ir-search\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Устройство: cuda
Путь до кэша: ../data/cache/hw5


## Данные MIRAGE

In [2]:
with open('../data/mirage/dataset.json', encoding='utf-8') as f:
    dataset = json.load(f)

with open('../data/mirage/doc_pool.json', encoding='utf-8') as f:
    doc_pool = json.load(f)

with open('../data/mirage/oracle.json', encoding='utf-8') as f:
    oracle = json.load(f)

query_map = {item['query_id']: item for item in dataset}

pool_by_qid = defaultdict(list)
for doc in doc_pool:
    pool_by_qid[doc['mapped_id']].append(doc)

qa_items = [
    (qid, query_map[qid]['query'], query_map[qid]['answer'])
    for qid in query_map
    if qid in oracle and qid in pool_by_qid
]

gold_answers = [answers for _, _, answers in qa_items]

print(f'Количество QA-примеров: {len(qa_items)}')
print(f'Пример вопроса: {qa_items[0][1]!r}')
print(f'Пример эталонного ответа: {qa_items[0][2]}')
print(f'Пример oracle-фрагмента: {oracle[qa_items[0][0]]["doc_chunk"][:200]!r}')

Количество QA-примеров: 7560
Пример вопроса: "What is John Mayne's occupation?"
Пример эталонного ответа: ['journalist', 'journo', 'journalists']
Пример oracle-фрагмента: 'Scottish printer, journalist and poet\nJohn Mayne (1759–1836) was a Scottish printer, journalist and poet born in Dumfries. In 1780, his poem "The Siller Gun" appeared in its original form in "Ruddiman'


## Вспомогательные функции

In [3]:
def normalize_answer(text: str) -> str:
    text = text.lower()
    text = re.sub(r'\b(a|an|the)\b', ' ', text)
    text = ''.join(ch for ch in text if ch not in string.punctuation)
    return ' '.join(text.split())


def get_tokens(text: str) -> list[str]:
    return normalize_answer(text).split()


def squad_em(prediction: str, gold_list: list[str]) -> int:
    norm_pred = normalize_answer(prediction)
    return int(any(norm_pred == normalize_answer(g) for g in gold_list))


def squad_f1(prediction: str, gold_list: list[str]) -> float:
    pred_tokens = get_tokens(prediction)
    best = 0.0

    for gold in gold_list:
        gold_tokens = get_tokens(gold)
        common = Counter(pred_tokens) & Counter(gold_tokens)
        overlap = sum(common.values())

        if overlap == 0:
            continue

        precision = overlap / len(pred_tokens) if pred_tokens else 0.0
        recall = overlap / len(gold_tokens) if gold_tokens else 0.0
        f1 = (
            2 * precision * recall / (precision + recall)
            if (precision + recall) > 0
            else 0.0
        )
        best = max(best, f1)

    return best


def mirage_em_loose(prediction: str, gold_list: list[str]) -> int:
    norm_pred = normalize_answer(prediction)

    for gold in gold_list:
        norm_gold = normalize_answer(gold)
        if norm_gold and (norm_gold in norm_pred or norm_pred in norm_gold):
            return 1
    return 0


def evaluate(predictions: list[str], gold_list: list[list[str]]) -> dict[str, float]:
    em_scores = [squad_em(p, g) for p, g in zip(predictions, gold_list)]
    f1_scores = [squad_f1(p, g) for p, g in zip(predictions, gold_list)]
    strict_scores = [squad_em(p, g) for p, g in zip(predictions, gold_list)]
    loose_scores = [mirage_em_loose(p, g) for p, g in zip(predictions, gold_list)]

    return {
        'EM': float(np.mean(em_scores)),
        'F1': float(np.mean(f1_scores)),
        'EM-strict': float(np.mean(strict_scores)),
        'EM-loose': float(np.mean(loose_scores)),
    }


def patched_sent_encode(tokenizer, sent):
    sent = sent.strip()

    if sent == '':
        return tokenizer.encode(
            '',
            add_special_tokens=True,
            max_length=getattr(tokenizer, 'model_max_length', None),
            truncation=True,
        )

    if isinstance(tokenizer, GPT2Tokenizer) or isinstance(tokenizer, RobertaTokenizer):
        return tokenizer.encode(
            sent,
            add_special_tokens=True,
            add_prefix_space=True,
            max_length=getattr(tokenizer, 'model_max_length', None),
            truncation=True,
        )

    return tokenizer.encode(
        sent,
        add_special_tokens=True,
        max_length=getattr(tokenizer, 'model_max_length', None),
        truncation=True,
    )


# костыль для совместимости transformers >=5.0 с bert_score
if version.parse(transformers.__version__) >= version.parse('5.0.0'):
    bs_utils.sent_encode = patched_sent_encode


def add_bertscore(
    metrics: dict[str, float], predictions: list[str], gold_list: list[list[str]]
) -> dict[str, float]:
    refs = [g[0] if g else '' for g in gold_list]
    preds = [p if p else '' for p in predictions]

    _, _, f1 = bert_score_fn(
        preds,
        refs,
        lang='en',
        verbose=False,
        device=DEVICE,
        use_fast_tokenizer=True,
    )

    out = dict(metrics)
    out['BERTScore-F1'] = float(f1.mean())
    return out


def format_metrics(metrics: dict[str, float]) -> dict[str, str]:
    return {k: f'{v:.4f}' for k, v in metrics.items()}

## Модель 1: экстрактивный QA

In [4]:
EXT_MODEL_NAME = 'deepset/roberta-base-squad2'

ext_tokenizer = AutoTokenizer.from_pretrained(EXT_MODEL_NAME)
ext_model = AutoModelForQuestionAnswering.from_pretrained(EXT_MODEL_NAME)

ext_pipe = pipeline(
    'question-answering',
    model=ext_model,
    tokenizer=ext_tokenizer,
    device=0 if DEVICE == 'cuda' else -1,
)

print('Экстрактивная модель:', EXT_MODEL_NAME)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 648.34it/s, Materializing param=roberta.encoder.layer.11.output.dense.weight]              
RobertaForQuestionAnswering LOAD REPORT from: deepset/roberta-base-squad2
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Экстрактивная модель: deepset/roberta-base-squad2


In [5]:
def run_extractive(inputs: list[dict], desc: str = 'Экстрактивный QA') -> list[str]:
    predictions = []
    batch_size = 64

    for start in tqdm(range(0, len(inputs), batch_size), desc=desc):
        batch = inputs[start : start + batch_size]

        questions = [item['question'] for item in batch]
        contexts = [item['context'] for item in batch]

        outputs = ext_pipe(question=questions, context=contexts)

        if isinstance(outputs, dict):
            outputs = [outputs]

        predictions.extend(output['answer'].strip() for output in outputs)

    return predictions


cache_key = 'ext_oracle'
ext_preds_oracle = cache_load(cache_key)

if ext_preds_oracle is None:
    oracle_inputs = [
        {'question': query, 'context': oracle[qid]['doc_chunk']}
        for qid, query, _ in qa_items
    ]
    ext_preds_oracle = run_extractive(oracle_inputs, desc='Экстрактивный QA: oracle')
    cache_save(cache_key, ext_preds_oracle)

metrics_ext_oracle = add_bertscore(
    evaluate(ext_preds_oracle, gold_answers),
    ext_preds_oracle,
    gold_answers,
)
print(
    'Результаты экстрактивного QA в oracle-режиме:',
    format_metrics(metrics_ext_oracle),
)

Loading weights: 100%|██████████| 389/389 [00:00<00:00, 598.51it/s, Materializing param=encoder.layer.23.output.dense.weight]              
RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.bias              | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Результаты экстрактивного QA в oracle-режиме: {'EM': '0.6392', 'F1': '0.7519', 'EM-strict': '0.6392', 'EM-loose': '0.8103', 'BERTScore-F1': '0.9522'}


## Модель 2: генеративный QA

In [6]:
GEN_MODEL_NAME = 'Qwen/Qwen2.5-1.5B-Instruct'

gen_tokenizer = AutoTokenizer.from_pretrained(
    GEN_MODEL_NAME,
    padding_side='left',
)

if gen_tokenizer.pad_token is None:
    gen_tokenizer.pad_token = gen_tokenizer.eos_token

gen_model = AutoModelForCausalLM.from_pretrained(
    GEN_MODEL_NAME,
    torch_dtype=torch.float16 if DEVICE == 'cuda' else torch.float32,
    device_map='auto' if DEVICE == 'cuda' else None,
)

if DEVICE != 'cuda':
    gen_model = gen_model.to(DEVICE)

gen_model.eval()

print('Генеративная модель:', GEN_MODEL_NAME)

`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 338/338 [00:01<00:00, 185.39it/s, Materializing param=model.norm.weight]                              


Генеративная модель: Qwen/Qwen2.5-1.5B-Instruct


In [7]:
def make_gen_prompt(question: str, context: str) -> str:
    return (
        'Answer the question concisely based on the context. '
        'Return only the short answer, without explanation.\n\n'
        f'Context: {context}\n\n'
        f'Question: {question}\n\n'
        'Answer:'
    )


def run_generative(
    qa_batch_items: list[tuple],
    get_context: Callable[[str, str], str],
    batch_size: int = 8,
    max_new_tokens: int = 32,
    desc: str = 'Генеративный QA',
) -> list[str]:
    predictions = []

    for start in tqdm(range(0, len(qa_batch_items), batch_size), desc=desc):
        batch = qa_batch_items[start : start + batch_size]
        prompts = [
            make_gen_prompt(question, get_context(qid, question))
            for qid, question, _ in batch
        ]

        enc = gen_tokenizer(
            prompts,
            return_tensors='pt',
            padding=True,
            truncation=True,
            max_length=1024,
        ).to(DEVICE)

        input_len = enc['input_ids'].shape[1]

        with torch.no_grad():
            output = gen_model.generate(
                **enc,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                pad_token_id=gen_tokenizer.eos_token_id,
            )

        for seq in output:
            answer = gen_tokenizer.decode(seq[input_len:], skip_special_tokens=True)
            answer = answer.split('\n')[0].strip()
            predictions.append(answer)

    return predictions


cache_key = 'gen_oracle'
gen_preds_oracle = cache_load(cache_key)

if gen_preds_oracle is None:
    gen_preds_oracle = run_generative(
        qa_items,
        get_context=lambda qid, _: oracle[qid]['doc_chunk'],
        desc='Генеративный QA: oracle',
    )
    cache_save(cache_key, gen_preds_oracle)

metrics_gen_oracle = add_bertscore(
    evaluate(gen_preds_oracle, gold_answers),
    gen_preds_oracle,
    gold_answers,
)
print(
    'Результаты генеративного QA в oracle-режиме:',
    format_metrics(metrics_gen_oracle),
)

Loading weights: 100%|██████████| 389/389 [00:01<00:00, 341.30it/s, Materializing param=encoder.layer.23.output.dense.weight]              
RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.bias              | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Результаты генеративного QA в oracle-режиме: {'EM': '0.4858', 'F1': '0.6608', 'EM-strict': '0.4858', 'EM-loose': '0.8634', 'BERTScore-F1': '0.9339'}


## Ретриверы

In [8]:
ce_ranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2', device=DEVICE)

print('Ретриверы: BM25 и CrossEncoder ms-marco-MiniLM-L-6-v2')

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 381.76it/s, Materializing param=classifier.weight]                                    
BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Ретриверы: BM25 и CrossEncoder ms-marco-MiniLM-L-6-v2


In [9]:
def rank_bm25(query: str, docs: list[dict]) -> list[dict]:
    corpus = [doc['doc_chunk'].split() for doc in docs]
    bm25 = BM25Okapi(corpus)
    scores = bm25.get_scores(query.split())
    order = np.argsort(scores)[::-1]
    return [docs[i] for i in order]


def rank_ce(query: str, docs: list[dict]) -> list[dict]:
    pairs = [(query, doc['doc_chunk']) for doc in docs]
    scores = ce_ranker.predict(pairs, show_progress_bar=False)
    order = np.argsort(scores)[::-1]
    return [docs[i] for i in order]


ranked_bm25 = cache_load('ranked_bm25')
ranked_ce = cache_load('ranked_ce')

if ranked_bm25 is None:
    ranked_bm25 = {
        qid: rank_bm25(question, pool_by_qid[qid])
        for qid, question, _ in tqdm(qa_items, desc='Ранжирование BM25')
    }
    cache_save('ranked_bm25', ranked_bm25)

if ranked_ce is None:
    ranked_ce = {
        qid: rank_ce(question, pool_by_qid[qid])
        for qid, question, _ in tqdm(qa_items, desc='Ранжирование CrossEncoder')
    }
    cache_save('ranked_ce', ranked_ce)


def oracle_recall_at_k(
    ranked: dict, oracle_data: dict, qa_batch_items: list[tuple], k: int = 1
) -> float:
    hits = 0
    for qid, _, _ in qa_batch_items:
        found = any(
            doc['doc_chunk'] == oracle_data[qid]['doc_chunk'] for doc in ranked[qid][:k]
        )
        hits += int(found)
    return hits / len(qa_batch_items)


print(
    'Полнота oracle-фрагмента для BM25 при k=1:',
    oracle_recall_at_k(ranked_bm25, oracle, qa_items, 1),
)
print(
    'Полнота oracle-фрагмента для BM25 при k=5:',
    oracle_recall_at_k(ranked_bm25, oracle, qa_items, 5),
)
print(
    'Полнота oracle-фрагмента для CrossEncoder при k=1:',
    oracle_recall_at_k(ranked_ce, oracle, qa_items, 1),
)
print(
    'Полнота oracle-фрагмента для CrossEncoder при k=5:',
    oracle_recall_at_k(ranked_ce, oracle, qa_items, 5),
)

Полнота oracle-фрагмента для BM25 при k=1: 0.4642857142857143
Полнота oracle-фрагмента для BM25 при k=5: 1.0
Полнота oracle-фрагмента для CrossEncoder при k=1: 0.8029100529100529
Полнота oracle-фрагмента для CrossEncoder при k=5: 1.0


## Эксперименты с top-1

In [10]:
def run_extractive_top1(
    qa_batch_items: list[tuple], ranked: dict, cache_key: str
) -> list[str]:
    predictions = cache_load(cache_key)

    if predictions is None:
        inputs = [
            {'question': question, 'context': ranked[qid][0]['doc_chunk']}
            for qid, question, _ in qa_batch_items
        ]
        predictions = run_extractive(inputs, desc=f'Экстрактивный QA: {cache_key}')
        cache_save(cache_key, predictions)

    return predictions


def run_generative_top1(
    qa_batch_items: list[tuple],
    ranked: dict,
    cache_key: str,
) -> list[str]:
    predictions = cache_load(cache_key)

    if predictions is None:
        predictions = run_generative(
            qa_batch_items,
            get_context=lambda qid, _: ranked[qid][0]['doc_chunk'],
            desc=f'Генеративный QA: {cache_key}',
        )
        cache_save(cache_key, predictions)

    return predictions


ext_preds_bm25_top1 = run_extractive_top1(qa_items, ranked_bm25, 'ext_bm25_top1')
ext_preds_ce_top1 = run_extractive_top1(qa_items, ranked_ce, 'ext_ce_top1')

metrics_ext_bm25_top1 = add_bertscore(
    evaluate(ext_preds_bm25_top1, gold_answers), ext_preds_bm25_top1, gold_answers
)
metrics_ext_ce_top1 = add_bertscore(
    evaluate(ext_preds_ce_top1, gold_answers), ext_preds_ce_top1, gold_answers
)

print(
    'Результаты экстрактивного QA с BM25 top-1:',
    format_metrics(metrics_ext_bm25_top1),
)
print(
    'Результаты экстрактивного QA с CrossEncoder top-1:',
    format_metrics(metrics_ext_ce_top1),
)


Loading weights: 100%|██████████| 389/389 [00:01<00:00, 386.91it/s, Materializing param=encoder.layer.23.output.dense.weight]              
RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.bias              | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Loading weights: 100%|██████████| 389/389 [00:01<00:00, 365.03it/s, Materializing param

Результаты экстрактивного QA с BM25 top-1: {'EM': '0.3265', 'F1': '0.4076', 'EM-strict': '0.3265', 'EM-loose': '0.4459', 'BERTScore-F1': '0.9009'}
Результаты экстрактивного QA с CrossEncoder top-1: {'EM': '0.5396', 'F1': '0.6403', 'EM-strict': '0.5396', 'EM-loose': '0.6880', 'BERTScore-F1': '0.9392'}


In [11]:
gen_preds_bm25_top1 = run_generative_top1(qa_items, ranked_bm25, 'gen_bm25_top1')
gen_preds_ce_top1 = run_generative_top1(qa_items, ranked_ce, 'gen_ce_top1')

metrics_gen_bm25_top1 = add_bertscore(
    evaluate(gen_preds_bm25_top1, gold_answers),
    gen_preds_bm25_top1,
    gold_answers,
)
metrics_gen_ce_top1 = add_bertscore(
    evaluate(gen_preds_ce_top1, gold_answers),
    gen_preds_ce_top1,
    gold_answers,
)

print(
    'Результаты генеративного QA с BM25 top-1:',
    format_metrics(metrics_gen_bm25_top1),
)
print(
    'Результаты генеративного QA с CrossEncoder top-1:',
    format_metrics(metrics_gen_ce_top1),
)

Loading weights: 100%|██████████| 389/389 [00:01<00:00, 333.33it/s, Materializing param=encoder.layer.23.output.dense.weight]              
RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.bias              | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Loading weights: 100%|██████████| 389/389 [00:01<00:00, 371.92it/s, Materializing param

Результаты генеративного QA с BM25 top-1: {'EM': '0.2464', 'F1': '0.3552', 'EM-strict': '0.2464', 'EM-loose': '0.4521', 'BERTScore-F1': '0.8921'}
Результаты генеративного QA с CrossEncoder top-1: {'EM': '0.4094', 'F1': '0.5629', 'EM-strict': '0.4094', 'EM-loose': '0.7287', 'BERTScore-F1': '0.9219'}


## Эксперименты с top-5 и mixed

In [12]:
def concat_docs(docs: list[dict], max_chars: int = 3000) -> str:
    text = '\n\n'.join(doc['doc_chunk'] for doc in docs)
    return text[:max_chars]


def run_extractive_top5_with_score(
    qa_batch_items: list[tuple],
    ranked: dict,
    cache_key: str,
    top_k: int = 5,
) -> list[str]:
    predictions = cache_load(cache_key)

    if predictions is not None:
        return predictions

    result = []

    for qid, question, _ in tqdm(qa_batch_items, desc=f'Экстрактивный QA: {cache_key}'):
        docs = ranked[qid][:top_k]
        best_score = -1.0
        best_answer = ''

        for doc in docs:
            output = ext_pipe(question=question, context=doc['doc_chunk'])
            if output['score'] > best_score:
                best_score = output['score']
                best_answer = output['answer'].strip()

        result.append(best_answer)

    cache_save(cache_key, result)
    return result


def run_generative_topk_concat(
    qa_batch_items: list[tuple],
    ranked: dict,
    cache_key: str,
    top_k: int = 5,
) -> list[str]:
    predictions = cache_load(cache_key)

    if predictions is not None:
        return predictions

    predictions = run_generative(
        qa_batch_items,
        get_context=lambda qid, _: concat_docs(ranked[qid][:top_k]),
        desc=f'Генеративный QA: {cache_key}',
    )
    cache_save(cache_key, predictions)
    return predictions


ext_preds_bm25_top5 = run_extractive_top5_with_score(
    qa_items,
    ranked_bm25,
    'ext_bm25_top5',
    top_k=5,
)
ext_preds_ce_top5 = run_extractive_top5_with_score(
    qa_items,
    ranked_ce,
    'ext_ce_top5',
    top_k=5,
)
ext_preds_mixed = run_extractive_top5_with_score(
    qa_items,
    pool_by_qid,
    'ext_mixed_pool',
    top_k=5,
)

metrics_ext_bm25_top5 = add_bertscore(
    evaluate(ext_preds_bm25_top5, gold_answers),
    ext_preds_bm25_top5,
    gold_answers,
)
metrics_ext_ce_top5 = add_bertscore(
    evaluate(ext_preds_ce_top5, gold_answers),
    ext_preds_ce_top5,
    gold_answers,
)
metrics_ext_mixed = add_bertscore(
    evaluate(ext_preds_mixed, gold_answers),
    ext_preds_mixed,
    gold_answers,
)

print(
    'Результаты экстрактивного QA с BM25 top-5:',
    format_metrics(metrics_ext_bm25_top5),
)
print(
    'Результаты экстрактивного QA с CrossEncoder top-5:',
    format_metrics(metrics_ext_ce_top5),
)
print(
    'Результаты экстрактивного QA с mixed-контекстом:',
    format_metrics(metrics_ext_mixed),
)

Loading weights: 100%|██████████| 389/389 [00:01<00:00, 369.91it/s, Materializing param=encoder.layer.23.output.dense.weight]              
RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.bias              | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Loading weights: 100%|██████████| 389/389 [00:01<00:00, 379.99it/s, Materializing param

Результаты экстрактивного QA с BM25 top-5: {'EM': '0.5216', 'F1': '0.6119', 'EM-strict': '0.5216', 'EM-loose': '0.6512', 'BERTScore-F1': '0.9357'}
Результаты экстрактивного QA с CrossEncoder top-5: {'EM': '0.5216', 'F1': '0.6119', 'EM-strict': '0.5216', 'EM-loose': '0.6512', 'BERTScore-F1': '0.9357'}
Результаты экстрактивного QA с mixed-контекстом: {'EM': '0.5216', 'F1': '0.6119', 'EM-strict': '0.5216', 'EM-loose': '0.6512', 'BERTScore-F1': '0.9357'}


In [13]:
gen_preds_bm25_top5 = run_generative_topk_concat(
    qa_items,
    ranked_bm25,
    'gen_bm25_top5',
    top_k=5,
)
gen_preds_ce_top5 = run_generative_topk_concat(
    qa_items,
    ranked_ce,
    'gen_ce_top5',
    top_k=5,
)

gen_preds_mixed = cache_load('gen_mixed_pool')
if gen_preds_mixed is None:
    gen_preds_mixed = run_generative(
        qa_items,
        get_context=lambda qid, _: concat_docs(pool_by_qid[qid], max_chars=3000),
        desc='Генеративный QA: mixed-контекст',
    )
    cache_save('gen_mixed_pool', gen_preds_mixed)

metrics_gen_bm25_top5 = add_bertscore(
    evaluate(gen_preds_bm25_top5, gold_answers),
    gen_preds_bm25_top5,
    gold_answers,
)
metrics_gen_ce_top5 = add_bertscore(
    evaluate(gen_preds_ce_top5, gold_answers),
    gen_preds_ce_top5,
    gold_answers,
)
metrics_gen_mixed = add_bertscore(
    evaluate(gen_preds_mixed, gold_answers),
    gen_preds_mixed,
    gold_answers,
)

print(
    'Результаты генеративного QA с BM25 top-5:',
    format_metrics(metrics_gen_bm25_top5),
)
print(
    'Результаты генеративного QA с CrossEncoder top-5:',
    format_metrics(metrics_gen_ce_top5),
)
print(
    'Результаты генеративного QA с mixed-контекстом:',
    format_metrics(metrics_gen_mixed),
)

Loading weights: 100%|██████████| 389/389 [00:01<00:00, 378.80it/s, Materializing param=encoder.layer.23.output.dense.weight]              
RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.bias              | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Loading weights: 100%|██████████| 389/389 [00:00<00:00, 660.22it/s, Materializing param

Результаты генеративного QA с BM25 top-5: {'EM': '0.2061', 'F1': '0.3567', 'EM-strict': '0.2061', 'EM-loose': '0.6470', 'BERTScore-F1': '0.8906'}
Результаты генеративного QA с CrossEncoder top-5: {'EM': '0.2594', 'F1': '0.4291', 'EM-strict': '0.2594', 'EM-loose': '0.7646', 'BERTScore-F1': '0.9015'}
Результаты генеративного QA с mixed-контекстом: {'EM': '0.2844', 'F1': '0.4480', 'EM-strict': '0.2844', 'EM-loose': '0.7544', 'BERTScore-F1': '0.9043'}


In [14]:
all_results = [
    ('RoBERTa (extractive)', 'Oracle', metrics_ext_oracle),
    ('RoBERTa (extractive)', 'BM25 top-1', metrics_ext_bm25_top1),
    ('RoBERTa (extractive)', 'CrossEncoder top-1', metrics_ext_ce_top1),
    ('RoBERTa (extractive)', 'BM25 top-5', metrics_ext_bm25_top5),
    ('RoBERTa (extractive)', 'CrossEncoder top-5', metrics_ext_ce_top5),
    ('RoBERTa (extractive)', 'Mixed', metrics_ext_mixed),
    ('Qwen2.5-1.5B (generative)', 'Oracle', metrics_gen_oracle),
    ('Qwen2.5-1.5B (generative)', 'BM25 top-1', metrics_gen_bm25_top1),
    ('Qwen2.5-1.5B (generative)', 'CrossEncoder top-1', metrics_gen_ce_top1),
    ('Qwen2.5-1.5B (generative)', 'BM25 top-5', metrics_gen_bm25_top5),
    ('Qwen2.5-1.5B (generative)', 'CrossEncoder top-5', metrics_gen_ce_top5),
    ('Qwen2.5-1.5B (generative)', 'Mixed', metrics_gen_mixed),
]

base_rows = []
for model_name, context_name, metrics in all_results:
    row = {
        'Модель': model_name,
        'Контекст': context_name,
    }
    row.update({k: f'{v:.4f}' for k, v in metrics.items()})
    base_rows.append(row)

results_df = pd.DataFrame(base_rows)
print('Сводная таблица базовых экспериментов:')
print(results_df.to_string(index=False))

Сводная таблица базовых экспериментов:
                   Модель           Контекст     EM     F1 EM-strict EM-loose BERTScore-F1
     RoBERTa (extractive)             Oracle 0.6392 0.7519    0.6392   0.8103       0.9522
     RoBERTa (extractive)         BM25 top-1 0.3265 0.4076    0.3265   0.4459       0.9009
     RoBERTa (extractive) CrossEncoder top-1 0.5396 0.6403    0.5396   0.6880       0.9392
     RoBERTa (extractive)         BM25 top-5 0.5216 0.6119    0.5216   0.6512       0.9357
     RoBERTa (extractive) CrossEncoder top-5 0.5216 0.6119    0.5216   0.6512       0.9357
     RoBERTa (extractive)              Mixed 0.5216 0.6119    0.5216   0.6512       0.9357
Qwen2.5-1.5B (generative)             Oracle 0.4858 0.6608    0.4858   0.8634       0.9339
Qwen2.5-1.5B (generative)         BM25 top-1 0.2464 0.3552    0.2464   0.4521       0.8921
Qwen2.5-1.5B (generative) CrossEncoder top-1 0.4094 0.5629    0.4094   0.7287       0.9219
Qwen2.5-1.5B (generative)         BM25 top-5 0.2061

## LLM-as-a-judge

In [15]:
JUDGE_PROMPT = (
    'You are an answer quality evaluator.\n'
    'Question: {question}\n'
    'Reference answers: {references}\n'
    'Predicted answer: {prediction}\n\n'
    'Is the predicted answer correct or semantically equivalent to at least one reference answer? '
    'Reply with exactly "yes" or "no".'
)


def judge_batch(
    qa_batch_items: list[tuple],
    predictions: list[str],
    batch_size: int = 8,
    desc: str = 'LLM-as-a-judge',
) -> list[int]:
    scores = []

    for start in tqdm(range(0, len(qa_batch_items), batch_size), desc=desc):
        batch_items = qa_batch_items[start : start + batch_size]
        batch_preds = predictions[start : start + batch_size]

        prompts = [
            JUDGE_PROMPT.format(
                question=question,
                references=' | '.join(answers),
                prediction=pred,
            )
            for (_, question, answers), pred in zip(batch_items, batch_preds)
        ]

        enc = gen_tokenizer(
            prompts,
            return_tensors='pt',
            padding=True,
            truncation=True,
            max_length=768,
        ).to(DEVICE)

        input_len = enc['input_ids'].shape[1]

        with torch.no_grad():
            output = gen_model.generate(
                **enc,
                max_new_tokens=4,
                do_sample=False,
                pad_token_id=gen_tokenizer.eos_token_id,
            )

        for seq in output:
            verdict = (
                gen_tokenizer.decode(seq[input_len:], skip_special_tokens=True)
                .strip()
                .lower()
            )
            scores.append(1 if verdict.startswith('yes') else 0)

    return scores

In [16]:
all_predictions_map = {
    'ext_oracle': ext_preds_oracle,
    'ext_bm25_top1': ext_preds_bm25_top1,
    'ext_ce_top1': ext_preds_ce_top1,
    'ext_bm25_top5': ext_preds_bm25_top5,
    'ext_ce_top5': ext_preds_ce_top5,
    'ext_mixed': ext_preds_mixed,
    'gen_oracle': gen_preds_oracle,
    'gen_bm25_top1': gen_preds_bm25_top1,
    'gen_ce_top1': gen_preds_ce_top1,
    'gen_bm25_top5': gen_preds_bm25_top5,
    'gen_ce_top5': gen_preds_ce_top5,
    'gen_mixed': gen_preds_mixed,
}

all_metrics_map = {
    'ext_oracle': metrics_ext_oracle,
    'ext_bm25_top1': metrics_ext_bm25_top1,
    'ext_ce_top1': metrics_ext_ce_top1,
    'ext_bm25_top5': metrics_ext_bm25_top5,
    'ext_ce_top5': metrics_ext_ce_top5,
    'ext_mixed': metrics_ext_mixed,
    'gen_oracle': metrics_gen_oracle,
    'gen_bm25_top1': metrics_gen_bm25_top1,
    'gen_ce_top1': metrics_gen_ce_top1,
    'gen_bm25_top5': metrics_gen_bm25_top5,
    'gen_ce_top5': metrics_gen_ce_top5,
    'gen_mixed': metrics_gen_mixed,
}

In [17]:
JUDGE_LIMIT = 5000

judge_rows = []

for label, preds in all_predictions_map.items():
    cache_key = f'judge_{label}'

    qa_subset = qa_items if JUDGE_LIMIT is None else qa_items[:JUDGE_LIMIT]
    pred_subset = preds if JUDGE_LIMIT is None else preds[:JUDGE_LIMIT]

    judge_scores = cache_load(cache_key)
    if judge_scores is None:
        judge_scores = judge_batch(
            qa_subset,
            pred_subset,
            desc=f'LLM-as-a-judge: {label}',
        )
        cache_save(cache_key, judge_scores)

    judge_mean = float(np.mean(judge_scores))
    metrics = all_metrics_map[label]

    judge_rows.append(
        {
            'Конфигурация': label,
            'LLM-judge': f'{judge_mean:.4f}',
            'EM': f'{metrics["EM"]:.4f}',
            'F1': f'{metrics["F1"]:.4f}',
            'EM-strict': f'{metrics["EM-strict"]:.4f}',
            'EM-loose': f'{metrics["EM-loose"]:.4f}',
            'BERTScore-F1': f'{metrics["BERTScore-F1"]:.4f}',
        }
    )

judge_df = pd.DataFrame(judge_rows)
print('Сравнение автоматических метрик и LLM-as-a-judge:')
print(judge_df.to_string(index=False))

LLM-as-a-judge: gen_mixed: 100%|██████████| 625/625 [02:57<00:00,  3.52it/s]

Сравнение автоматических метрик и LLM-as-a-judge:
 Конфигурация LLM-judge     EM     F1 EM-strict EM-loose BERTScore-F1
   ext_oracle    0.1312 0.6392 0.7519    0.6392   0.8103       0.9522
ext_bm25_top1    0.0780 0.3265 0.4076    0.3265   0.4459       0.9009
  ext_ce_top1    0.1128 0.5396 0.6403    0.5396   0.6880       0.9392
ext_bm25_top5    0.1078 0.5216 0.6119    0.5216   0.6512       0.9357
  ext_ce_top5    0.1078 0.5216 0.6119    0.5216   0.6512       0.9357
    ext_mixed    0.1078 0.5216 0.6119    0.5216   0.6512       0.9357
   gen_oracle    0.2546 0.4858 0.6608    0.4858   0.8634       0.9339
gen_bm25_top1    0.1564 0.2464 0.3552    0.2464   0.4521       0.8921
  gen_ce_top1    0.2232 0.4094 0.5629    0.4094   0.7287       0.9219
gen_bm25_top5    0.1684 0.2061 0.3567    0.2061   0.6470       0.8906
  gen_ce_top5    0.1884 0.2594 0.4291    0.2594   0.7646       0.9015
    gen_mixed    0.2050 0.2844 0.4480    0.2844   0.7544       0.9043


## Результаты

In [18]:
full_rows = []
for model_name, context_name, metrics in all_results:
    row = {
        'Модель': model_name,
        'Контекст': context_name,
    }
    row.update({k: f'{v:.4f}' for k, v in metrics.items()})
    full_rows.append(row)

full_df = pd.DataFrame(full_rows)
print('Итоговая таблица автоматических метрик:')
print(full_df.to_string(index=False))

Итоговая таблица автоматических метрик:
                   Модель           Контекст     EM     F1 EM-strict EM-loose BERTScore-F1
     RoBERTa (extractive)             Oracle 0.6392 0.7519    0.6392   0.8103       0.9522
     RoBERTa (extractive)         BM25 top-1 0.3265 0.4076    0.3265   0.4459       0.9009
     RoBERTa (extractive) CrossEncoder top-1 0.5396 0.6403    0.5396   0.6880       0.9392
     RoBERTa (extractive)         BM25 top-5 0.5216 0.6119    0.5216   0.6512       0.9357
     RoBERTa (extractive) CrossEncoder top-5 0.5216 0.6119    0.5216   0.6512       0.9357
     RoBERTa (extractive)              Mixed 0.5216 0.6119    0.5216   0.6512       0.9357
Qwen2.5-1.5B (generative)             Oracle 0.4858 0.6608    0.4858   0.8634       0.9339
Qwen2.5-1.5B (generative)         BM25 top-1 0.2464 0.3552    0.2464   0.4521       0.8921
Qwen2.5-1.5B (generative) CrossEncoder top-1 0.4094 0.5629    0.4094   0.7287       0.9219
Qwen2.5-1.5B (generative)         BM25 top-5 0.206

In [19]:
def best_row_by_metric(df: pd.DataFrame, metric: str) -> pd.Series:
    numeric = df.copy()
    numeric[metric] = numeric[metric].astype(float)
    return numeric.sort_values(metric, ascending=False).iloc[0]


best_em = best_row_by_metric(full_df, 'EM')
best_f1 = best_row_by_metric(full_df, 'F1')
best_bert = best_row_by_metric(full_df, 'BERTScore-F1')

print('Лучший результат по EM:')
print(best_em.to_string())
print()

print('Лучший результат по F1:')
print(best_f1.to_string())
print()

print('Лучший результат по BERTScore-F1:')
print(best_bert.to_string())

Лучший результат по EM:
Модель          RoBERTa (extractive)
Контекст                      Oracle
EM                            0.6392
F1                            0.7519
EM-strict                     0.6392
EM-loose                      0.8103
BERTScore-F1                  0.9522

Лучший результат по F1:
Модель          RoBERTa (extractive)
Контекст                      Oracle
EM                            0.6392
F1                            0.7519
EM-strict                     0.6392
EM-loose                      0.8103
BERTScore-F1                  0.9522

Лучший результат по BERTScore-F1:
Модель          RoBERTa (extractive)
Контекст                      Oracle
EM                            0.6392
F1                            0.7519
EM-strict                     0.6392
EM-loose                      0.8103
BERTScore-F1                  0.9522
